# 实验4.1 常见轻量化深度学习网络实验

> **实验名称**：Lab4.1 常见轻量化深度学习网络实验
> **运行环境**：GitCode CANNLab 云沙箱（昇腾 Atlas NPU · Ascend 910B3）| cann_9.0.0-py3.11-A2-arm-20260715
> **建议学时**：4 学时

## 实验导学

在深度学习实际部署中，模型往往面临**计算资源有限、存储空间受限、功耗敏感**等挑战。轻量化网络通过精巧的架构设计，在保持较高精度的同时大幅减少参数量和计算量，是边缘端和移动端 AI 部署的关键技术。

本 Notebook 以**猫狗分类**为应用案例，依次使用 4 种经典网络进行实验，并对它们进行对比分析：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">网络</th>
<th style="text-align: left;">年份</th>
<th style="text-align: left;">核心思想</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><strong>ResNet18</strong></td>
<td style="text-align: left;">2015</td>
<td style="text-align: left;">残差连接（Residual Connection）</td>
<td style="text-align: left;">解决深层网络梯度消失，结构经典，作为基准</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><strong>MobileNetV2</strong></td>
<td style="text-align: left;">2018</td>
<td style="text-align: left;">深度可分离卷积 + 倒残差结构</td>
<td style="text-align: left;">面向移动端，参数极少</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><strong>ShuffleNetV2</strong></td>
<td style="text-align: left;">2018</td>
<td style="text-align: left;">通道重排 + 分组卷积</td>
<td style="text-align: left;">面向端侧，计算效率极高</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;"><strong>EfficientNet-B0</strong></td>
<td style="text-align: left;">2019</td>
<td style="text-align: left;">复合缩放（Compound Scaling）</td>
<td style="text-align: left;">精度与效率的最优平衡</td>
</tr>
</table>

## 学习目标

1. 理解 ResNet18 及 3 种轻量化网络（MobileNet、ShuffleNet、EfficientNet）的设计原理
2. 掌握迁移学习（Transfer Learning）的方法：加载预训练权重 + 替换分类层 + 微调
3. 在昇腾 NPU 上完成 4 种网络的训练与推理，体验 AI 计算加速
4. 对比分析 4 种网络在**参数量、模型体积、训练时间、推理速度、分类精度**上的差异
5. 理解轻量化网络设计的核心思想：深度可分离卷积、分组卷积、通道重排、复合缩放

## 实验流程

```
环境准备 → 数据加载与增强 → 工具函数定义
  → ResNet18 训练与评估
  → MobileNetV2 训练与评估
  → ShuffleNetV2 训练与评估
  → EfficientNet-B0 训练与评估
  → 四种网络对比分析 → 课后练习
```

> **说明**：由于教学需要快速得到运行结果，本实验仅使用 4 张图片（cat1.jpg、cat2.jpg、dog1.jpg、dog2.jpg），通过数据增强扩充训练集。采用预训练权重 + 微调的策略，使模型在少量数据上也能快速收敛。所有图表使用英文标注以避免中文乱码。

---

## 一、环境准备

首先导入所需的库，并检测昇腾 NPU 是否可用。本实验在 GitCode CANNLab 云沙箱上运行，使用 Ascend 910B3 NPU 进行加速。

**代码说明**：
- 导入 PyTorch、torchvision、matplotlib 等核心库。
- 尝试导入 `torch_npu`（昇腾 NPU 适配层），若可用则将计算设备设为 NPU，否则回退到 CPU。
- 创建 `output` 目录用于保存实验结果图表。
- 设置 matplotlib 样式，确保图表清晰。

**预期结果**：
- 打印 PyTorch 和 torchvision 版本号
- 打印计算设备（`npu` 或 `cpu`）
- 若 NPU 可用，打印 NPU 型号（如 `Ascend910B3`）

In [ ]:
import os
import time
import io
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 尝试导入 torch_npu（昇腾 NPU 适配层）
try:
    import torch_npu
    HAS_NPU = True
except ImportError:
    HAS_NPU = False

# 设置计算设备
if HAS_NPU and torch.npu.is_available():
    device = torch.device('npu')
else:
    device = torch.device('cpu')

# 创建输出目录
os.makedirs('./output', exist_ok=True)

# 打印环境信息
print(f"PyTorch version     : {torch.__version__}")
print(f"torchvision version : {torchvision.__version__}")
print(f"Device              : {device}")
if HAS_NPU:
    print(f"torch_npu version   : {torch_npu.__version__}")
    if torch.npu.is_available():
        print(f"NPU model           : {torch.npu.get_device_name(0)}")
else:
    print("torch_npu           : not installed (will use CPU)")

# 设置随机种子保证可复现
torch.manual_seed(42)
np.random.seed(42)
print("\nEnvironment ready.")

## 二、数据准备

### 2.1 数据集介绍

本实验使用 `images` 文件夹中的 4 张图片进行猫狗分类：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">图片</th>
<th style="text-align: left;">类别</th>
<th style="text-align: left;">标签</th>
</tr>
<tr>
<td style="text-align: left;">cat1.jpg</td>
<td style="text-align: left;">猫</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">cat2.jpg</td>
<td style="text-align: left;">猫</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">dog1.jpg</td>
<td style="text-align: left;">狗</td>
<td style="text-align: left;">1</td>
</tr>
<tr>
<td style="text-align: left;">dog2.jpg</td>
<td style="text-align: left;">狗</td>
<td style="text-align: left;">1</td>
</tr>
</table>

由于只有 4 张图片，直接训练难以收敛。我们采用以下策略：
1. **加载 ImageNet 预训练权重**：模型已学到丰富的图像特征，只需微调即可。
2. **数据增强**：对每张图片进行随机裁剪、翻转、旋转、颜色抖动等增强，扩充训练集。
3. **快速训练**：仅训练少量 epoch，快速得到结果。

**代码说明**：
- 使用 `PIL.Image` 加载图片并展示。
- 将 4 张原始图片绘制在一起，保存到 `output/dataset_samples.png`。

**预期结果**：
- 显示 4 张图片（2 只猫 + 2 只狗）
- 图片保存到 `output/dataset_samples.png`

In [ ]:
# 可视化 4 张原始图片
image_dir = './images'
image_files = ['cat1.jpg', 'cat2.jpg', 'dog1.jpg', 'dog2.jpg']
titles = ['Cat 1 (label=0)', 'Cat 2 (label=0)', 'Dog 1 (label=1)', 'Dog 2 (label=1)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_file, title in zip(axes, image_files, titles):
    img = Image.open(os.path.join(image_dir, img_file))
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Cat and Dog Classification Dataset (4 images)', fontsize=14)
plt.tight_layout()
plt.savefig('./output/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dataset visualization saved to ./output/dataset_samples.png")

### 2.2 数据增强与 DataLoader 构建

**数据增强策略详解**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">增强方式</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>Resize(256)</code></td>
<td style="text-align: left;">缩放到 256×256</td>
<td style="text-align: left;">统一尺寸</td>
</tr>
<tr>
<td style="text-align: left;"><code>RandomCrop(224)</code></td>
<td style="text-align: left;">随机裁剪到 224×224</td>
<td style="text-align: left;">模拟不同位置，增加位置多样性</td>
</tr>
<tr>
<td style="text-align: left;"><code>RandomHorizontalFlip</code></td>
<td style="text-align: left;">随机水平翻转</td>
<td style="text-align: left;">增加方向多样性</td>
</tr>
<tr>
<td style="text-align: left;"><code>RandomRotation(15)</code></td>
<td style="text-align: left;">随机旋转 ±15°</td>
<td style="text-align: left;">增加角度多样性</td>
</tr>
<tr>
<td style="text-align: left;"><code>ColorJitter</code></td>
<td style="text-align: left;">颜色抖动</td>
<td style="text-align: left;">增加光照颜色多样性</td>
</tr>
</table>

通过将每张图片增强 50 次，4 张图片扩充为 200 个训练样本。测试集使用原始 4 张图片（仅 Resize，不增强）。

**代码说明**：
- `CatDogDataset`：自定义数据集类，支持数据增强倍数 `augment_times`。
- `train_transform`：训练时的数据增强流水线。
- `test_transform`：测试时的预处理流水线（无随机增强）。
- 训练集 200 个样本（4×50），测试集 4 个样本。

**预期结果**：
- `Training samples: 200`
- `Test samples: 4`

In [ ]:
# 训练时的数据增强流水线
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 测试时的预处理流水线（无随机增强）
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 自定义猫狗分类数据集
class CatDogDataset(Dataset):
    """猫狗分类数据集
    
    Args:
        image_dir: 图片目录
        augment_times: 每张图片的增强次数（1 表示不增强）
        transform: 数据预处理/增强流水线
    """
    def __init__(self, image_dir, augment_times=1, transform=None):
        self.transform = transform
        self.samples = []
        # cat -> label 0, dog -> label 1
        file_label_map = {
            'cat1.jpg': 0, 'cat2.jpg': 0,
            'dog1.jpg': 1, 'dog2.jpg': 1
        }
        for fname, label in file_label_map.items():
            path = os.path.join(image_dir, fname)
            for _ in range(augment_times):
                self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# 创建数据集和 DataLoader
train_dataset = CatDogDataset(image_dir, augment_times=50, transform=train_transform)
test_dataset  = CatDogDataset(image_dir, augment_times=1,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=4,  shuffle=False)

print(f"Training samples : {len(train_dataset)}")
print(f"Test samples     : {len(test_dataset)}")
print(f"Number of classes: 2 (cat=0, dog=1)")

## 三、公共工具函数

定义 5 种网络共用的训练、评估和可视化工具函数。

**函数说明**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">函数</th>
<th style="text-align: left;">功能</th>
</tr>
<tr>
<td style="text-align: left;"><code>train_model()</code></td>
<td style="text-align: left;">训练模型，记录每个 epoch 的 loss 和 accuracy，返回训练历史</td>
</tr>
<tr>
<td style="text-align: left;"><code>evaluate_model()</code></td>
<td style="text-align: left;">在测试集上评估模型，返回 loss 和 accuracy</td>
</tr>
<tr>
<td style="text-align: left;"><code>count_parameters()</code></td>
<td style="text-align: left;">统计模型可训练参数量</td>
</tr>
<tr>
<td style="text-align: left;"><code>get_model_size_mb()</code></td>
<td style="text-align: left;">计算模型 state_dict 的存储大小（MB）</td>
</tr>
<tr>
<td style="text-align: left;"><code>measure_inference_time()</code></td>
<td style="text-align: left;">测量模型平均推理时间（ms）</td>
</tr>
<tr>
<td style="text-align: left;"><code>plot_training_curves()</code></td>
<td style="text-align: left;">绘制训练过程的 loss 和 accuracy 曲线</td>
</tr>
</table>

**训练策略**：
- 损失函数：`CrossEntropyLoss`（交叉熵损失，适用于分类任务）
- 优化器：`Adam`（自适应学习率，收敛快）
- 学习率：`0.001`（微调时常用值）
- epoch 数：`5`（少量 epoch 快速训练）

**预期结果**：打印 `Utility functions ready.`

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=5,
                 learning_rate=0.001, model_name='Model'):
    """训练模型并返回训练历史记录
    
    Args:
        model: 待训练的模型
        train_loader: 训练数据 DataLoader
        test_loader: 测试数据 DataLoader
        num_epochs: 训练轮数
        learning_rate: 学习率
        model_name: 模型名称（用于打印）
    Returns:
        history: dict，包含每个 epoch 的 train/test loss 和 accuracy
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'training_time': 0.0
    }

    start_time = time.time()

    for epoch in range(num_epochs):
        # === 训练阶段 ===
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            running_correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        # === 测试阶段 ===
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                test_correct += predicted.eq(labels).sum().item()
                test_total += labels.size(0)

        test_loss = test_loss / test_total
        test_acc = test_correct / test_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        print(f'  [{model_name}] Epoch {epoch+1}/{num_epochs}: '
              f'Train Loss={train_loss:.4f} Acc={train_acc:.4f} | '
              f'Test Loss={test_loss:.4f} Acc={test_acc:.4f}')

    elapsed = time.time() - start_time
    history['training_time'] = elapsed
    print(f'  [{model_name}] Training completed in {elapsed:.2f}s')
    return history


def count_parameters(model):
    """统计模型可训练参数总量"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model):
    """计算模型 state_dict 的存储大小（MB）"""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024 / 1024


def measure_inference_time(model, test_loader, num_runs=20):
    """测量模型平均推理时间（ms），多次运行取平均"""
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            for images, _ in test_loader:
                images = images.to(device)
                if device.type == 'npu':
                    torch.npu.synchronize()
                start = time.time()
                _ = model(images)
                if device.type == 'npu':
                    torch.npu.synchronize()
                times.append(time.time() - start)
    return np.mean(times) * 1000  # 转换为毫秒


def plot_training_curves(history, model_name, save_dir='./output'):
    """绘制训练过程的 loss 和 accuracy 曲线"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss 曲线
    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=5)
    ax1.plot(epochs, history['test_loss'], 'r-s', label='Test Loss', markersize=5)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'{model_name} - Loss', fontsize=13)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Accuracy 曲线
    ax2.plot(epochs, history['train_acc'], 'b-o', label='Train Accuracy', markersize=5)
    ax2.plot(epochs, history['test_acc'], 'r-s', label='Test Accuracy', markersize=5)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title(f'{model_name} - Accuracy', fontsize=13)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = f'{save_dir}/{model_name}_training_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Training curves saved to {save_path}")

print("Utility functions ready.")

## 四、ResNet18 — 残差网络（基准模型）

### 4.1 ResNet 原理

ResNet（Residual Network）由何恺明等人于 2015 年提出，核心创新是**残差连接（Residual Connection）**。

**深层网络的痛点**：随着网络加深，梯度在反向传播中会逐渐消失或爆炸，导致深层网络反而比浅层网络更难训练——这被称为**网络退化问题**。

**残差连接的解决方案**：不直接学习目标映射 $H(x)$，而是学习**残差** $F(x) = H(x) - x$，然后通过**快捷连接（Shortcut Connection）**将输入 $x$ 直接加到输出：

$$H(x) = F(x) + x$$

这样梯度可以通过快捷连接直接传回浅层，缓解梯度消失问题，使训练上百层的网络成为可能。

**ResNet18 结构**：
- 包含 4 个 stage，每个 stage 有 2 个 BasicBlock（含 2 个 3×3 卷积）
- 每个 stage 之间通过 stride=2 下采样
- 最终全局平均池化 + 全连接层输出分类结果
- 总共 18 层（1 个初始卷积 + 16 个残差块卷积 + 1 个全连接）

<img src="./images/resnet18_structure.png" alt="ResNet18 Structure" width="600" style="display: block; margin-left: 0;" />

> **为什么选 ResNet18 作为基准？** ResNet18 是 ResNet 家族中最小的模型，参数量约 11.7M，结构简单清晰，训练快速，适合作为教学基准模型。其他轻量化网络将与它对比参数量、速度和精度。

### 4.2 构建 ResNet18 模型

**代码说明**：
- 使用 `torchvision.models.resnet18` 加载 ImageNet 预训练权重（`IMAGENET1K_V1`）。
- 将最后的全连接层 `model.fc` 替换为 `nn.Linear(512, 2)`，输出 2 个类别（猫和狗）。
- 将模型移至 NPU 设备。
- 打印模型参数量和模型大小。

**预期结果**：
- `ResNet18 parameters: ~11,176,450`
- `ResNet18 model size: ~43 MB`

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# 加载 ImageNet 预训练 ResNet18
model_resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# 替换最后的全连接层：1000 类 -> 2 类（猫/狗）
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 2)
model_resnet = model_resnet.to(device)

# 统计参数量和模型大小
resnet_params = count_parameters(model_resnet)
resnet_size = get_model_size_mb(model_resnet)
print(f"ResNet18 parameters : {resnet_params:,}")
print(f"ResNet18 model size : {resnet_size:.2f} MB")
print(f"Final layer         : {model_resnet.fc}")

### 4.3 训练 ResNet18

**代码说明**：
- 调用 `train_model()` 函数训练 5 个 epoch。
- 使用学习率 `0.001`，Adam 优化器。
- 训练过程中打印每个 epoch 的 loss 和 accuracy。
- 训练完成后绘制 loss 和 accuracy 曲线。

**预期结果**：
- 训练 loss 逐渐下降，accuracy 逐渐上升
- 5 个 epoch 后测试准确率应接近 100%（4 张图片分类简单）
- 训练曲线保存到 `output/ResNet18_training_curves.png`

In [ ]:
# 训练 ResNet18
print("Training ResNet18...")
history_resnet = train_model(
    model_resnet, train_loader, test_loader,
    num_epochs=5, learning_rate=0.001, model_name='ResNet18'
)

# 绘制训练曲线
plot_training_curves(history_resnet, 'ResNet18')

In [ ]:
# 评估 ResNet18：推理时间和测试准确率
resnet_inference_time = measure_inference_time(model_resnet, test_loader)
resnet_test_acc = history_resnet['test_acc'][-1]
print(f"ResNet18 test accuracy     : {resnet_test_acc*100:.1f}%")
print(f"ResNet18 inference time    : {resnet_inference_time:.2f} ms")
print(f"ResNet18 training time     : {history_resnet['training_time']:.2f} s")

## 五、MobileNetV2 — 深度可分离卷积网络

### 5.1 MobileNet 原理

MobileNet 由 Google 于 2017 年提出，专为移动端和嵌入式设备设计。其核心创新是**深度可分离卷积（Depthwise Separable Convolution）**。

**标准卷积 vs 深度可分离卷积**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">类型</th>
<th style="text-align: left;">操作</th>
<th style="text-align: left;">计算量</th>
</tr>
<tr>
<td style="text-align: left;">标准卷积</td>
<td style="text-align: left;">输入 $C_{in}$ 通道 → 输出 $C_{out}$ 通道，每个输出通道与所有输入通道卷积</td>
<td style="text-align: left;">$D_K^2 \cdot C_{in} \cdot C_{out} \cdot D_F^2$</td>
</tr>
<tr>
<td style="text-align: left;">深度可分离卷积</td>
<td style="text-align: left;">① 逐通道卷积（Depthwise）：每个输入通道单独卷积<br>② 逐点卷积（Pointwise）：1×1 卷积调整通道数</td>
<td style="text-align: left;">$D_K^2 \cdot C_{in} \cdot D_F^2 + C_{in} \cdot C_{out} \cdot D_F^2$</td>
</tr>
</table>

其中 $D_K$ 为卷积核尺寸，$D_F$ 为特征图尺寸。深度可分离卷积的计算量约为标准卷积的 $\frac{1}{C_{out}} + \frac{1}{D_K^2}$，以 3×3 卷积、256 输出通道为例，计算量减少约 8~9 倍。

**MobileNetV2 的额外创新 — 倒残差结构（Inverted Residual）**：
- 先用 1×1 卷积**扩展**通道数（低维 → 高维）
- 在高维空间做 Depthwise 卷积提取特征
- 再用 1×1 卷积**压缩**通道数（高维 → 低维）
- 形成"瘦→胖→瘦"的倒残差块，与 ResNet 的"胖→瘦→胖"恰好相反

### 5.2 构建 MobileNetV2 模型

**代码说明**：
- 使用 `torchvision.models.mobilenet_v2` 加载预训练权重。
- MobileNetV2 的分类器结构为 `classifier = Sequential(Dropout, Linear)`，替换 `classifier[1]` 为 `nn.Linear(1280, 2)`。
- 打印参数量和模型大小，与 ResNet18 对比。

**预期结果**：
- `MobileNetV2 parameters: ~3.5M`（约为 ResNet18 的 1/3）
- `MobileNetV2 model size: ~14 MB`

In [ ]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# 加载 ImageNet 预训练 MobileNetV2
model_mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)

# 替换分类器最后一层：1000 类 -> 2 类
model_mobilenet.classifier[1] = nn.Linear(model_mobilenet.classifier[1].in_features, 2)
model_mobilenet = model_mobilenet.to(device)

# 统计参数量和模型大小
mobilenet_params = count_parameters(model_mobilenet)
mobilenet_size = get_model_size_mb(model_mobilenet)
print(f"MobileNetV2 parameters : {mobilenet_params:,}")
print(f"MobileNetV2 model size : {mobilenet_size:.2f} MB")
print(f"Final layer            : {model_mobilenet.classifier[1]}")

### 5.3 训练 MobileNetV2

**代码说明**：
- 同样训练 5 个 epoch，学习率 0.001。
- 观察训练曲线与 ResNet18 的差异。

**预期结果**：
- 训练速度比 ResNet18 更快（参数更少）
- 测试准确率应接近 100%
- 训练曲线保存到 `output/MobileNetV2_training_curves.png`

In [ ]:
# 训练 MobileNetV2
print("Training MobileNetV2...")
history_mobilenet = train_model(
    model_mobilenet, train_loader, test_loader,
    num_epochs=5, learning_rate=0.001, model_name='MobileNetV2'
)

# 绘制训练曲线
plot_training_curves(history_mobilenet, 'MobileNetV2')

In [ ]:
# 评估 MobileNetV2
mobilenet_inference_time = measure_inference_time(model_mobilenet, test_loader)
mobilenet_test_acc = history_mobilenet['test_acc'][-1]
print(f"MobileNetV2 test accuracy     : {mobilenet_test_acc*100:.1f}%")
print(f"MobileNetV2 inference time    : {mobilenet_inference_time:.2f} ms")
print(f"MobileNetV2 training time     : {history_mobilenet['training_time']:.2f} s")

## 六、ShuffleNetV2 — 通道重排网络

### 6.1 ShuffleNet 原理

ShuffleNet 由旷视科技（Megvii）于 2018 年提出，核心创新是**通道重排（Channel Shuffle）+ 分组卷积（Group Convolution）**。

**分组卷积的痛点**：分组卷积将通道分成若干组，每组独立卷积，能减少计算量。但组间信息不流通，阻碍了特征融合。

**通道重排的解决方案**：在两个分组卷积之间插入通道重排操作，将各组通道交错排列，使下一层的分组卷积能接收来自所有组的信息：

```
Group 1: [A B C]     Group 1: [A D G]  (重排后)
Group 2: [D E F] --> Group 2: [B E H]
Group 3: [G H I]     Group 3: [C F I]
```

**ShuffleNetV2 的设计准则**（4 条实用指导原则）：
1. 输入输出通道数相等时，内存访问开销最小
2. 过量的分组卷积会增加内存访问开销
3. 网络碎片化（多分支）会降低并行度
4. Element-wise 操作（如 ReLU、Add）不可忽视

基于这些准则设计了 ShuffleNetV2 的基本单元，采用**通道分割（Channel Split）**代替分组卷积，更加高效。

### 6.2 构建 ShuffleNetV2 模型

**代码说明**：
- 使用 `torchvision.models.shufflenet_v2_x0_5`（缩放系数 0.5，最小的版本）。
- 替换最后的全连接层为 `nn.Linear(1024, 2)`。
- ShuffleNetV2 的参数量极少，适合端侧部署。

**预期结果**：
- `ShuffleNetV2 parameters: ~1.4M`（5 种网络中最少之一）
- `ShuffleNetV2 model size: ~5 MB`

In [ ]:
from torchvision.models import shufflenet_v2_x0_5, ShuffleNet_V2_X0_5_Weights

# 加载 ImageNet 预训练 ShuffleNetV2 (scale=0.5)
model_shufflenet = shufflenet_v2_x0_5(weights=ShuffleNet_V2_X0_5_Weights.IMAGENET1K_V1)

# 替换最后的全连接层：1000 类 -> 2 类
model_shufflenet.fc = nn.Linear(model_shufflenet.fc.in_features, 2)
model_shufflenet = model_shufflenet.to(device)

# 统计参数量和模型大小
shufflenet_params = count_parameters(model_shufflenet)
shufflenet_size = get_model_size_mb(model_shufflenet)
print(f"ShuffleNetV2 parameters : {shufflenet_params:,}")
print(f"ShuffleNetV2 model size : {shufflenet_size:.2f} MB")
print(f"Final layer             : {model_shufflenet.fc}")

### 6.3 训练 ShuffleNetV2

**代码说明**：
- 同样训练 5 个 epoch，学习率 0.001。
- ShuffleNetV2 参数极少，训练应最快。

**预期结果**：
- 训练速度最快（参数最少）
- 测试准确率应接近 100%
- 训练曲线保存到 `output/ShuffleNetV2_training_curves.png`

In [ ]:
# 训练 ShuffleNetV2
print("Training ShuffleNetV2...")
history_shufflenet = train_model(
    model_shufflenet, train_loader, test_loader,
    num_epochs=5, learning_rate=0.001, model_name='ShuffleNetV2'
)

# 绘制训练曲线
plot_training_curves(history_shufflenet, 'ShuffleNetV2')

In [ ]:
# 评估 ShuffleNetV2
shufflenet_inference_time = measure_inference_time(model_shufflenet, test_loader)
shufflenet_test_acc = history_shufflenet['test_acc'][-1]
print(f"ShuffleNetV2 test accuracy     : {shufflenet_test_acc*100:.1f}%")
print(f"ShuffleNetV2 inference time    : {shufflenet_inference_time:.2f} ms")
print(f"ShuffleNetV2 training time     : {history_shufflenet['training_time']:.2f} s")

## 七、EfficientNet-B0 — 复合缩放网络

### 7.1 EfficientNet 原理

EfficientNet 由 Google 于 2019 年提出，核心创新是**复合缩放（Compound Scaling）**方法。

**传统缩放方法的局限**：提升模型性能有三种维度：
- **深度（Depth）**：增加网络层数
- **宽度（Width）**：增加每层通道数
- **分辨率（Resolution）**：增大输入图像尺寸

传统方法通常只缩放其中一个维度（如只加深或只加宽），难以充分利用三者协同效应。

**复合缩放的解决方案**：用一个统一的缩放系数 $\phi$（phi）同时缩放三个维度：

$$d = \alpha^\phi, \quad w = \beta^\phi, \quad r = \gamma^\phi$$

约束条件：$\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$（即 FLOPS 增加约 $2^\phi$ 倍）

通过神经架构搜索（NAS）找到最优的基线模型 **EfficientNet-B0**，然后通过不同 $\phi$ 值得到 B1~B7 系列。

**EfficientNet-B0 的结构特点**：
- 基于 **MBConv**（Mobile Inverted Bottleneck Convolution）模块
- 使用 **Swish** 激活函数：$f(x) = x \cdot \text{sigmoid}(x)$
- 引入 **Squeeze-and-Excitation（SE）** 注意力模块
- B0 是最小的基线模型，$\phi=0$

### 7.2 构建 EfficientNet-B0 模型

**代码说明**：
- 使用 `torchvision.models.efficientnet_b0` 加载预训练权重。
- EfficientNet 的分类器结构为 `classifier = Sequential(Dropout, Linear)`，替换 `classifier[1]` 为 `nn.Linear(1280, 2)`。

**预期结果**：
- `EfficientNet-B0 parameters: ~5.3M`
- `EfficientNet-B0 model size: ~20 MB`

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# 加载 ImageNet 预训练 EfficientNet-B0
model_efficientnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# 替换分类器最后一层：1000 类 -> 2 类
model_efficientnet.classifier[1] = nn.Linear(model_efficientnet.classifier[1].in_features, 2)
model_efficientnet = model_efficientnet.to(device)

# 统计参数量和模型大小
efficientnet_params = count_parameters(model_efficientnet)
efficientnet_size = get_model_size_mb(model_efficientnet)
print(f"EfficientNet-B0 parameters : {efficientnet_params:,}")
print(f"EfficientNet-B0 model size : {efficientnet_size:.2f} MB")
print(f"Final layer                : {model_efficientnet.classifier[1]}")

### 7.3 训练 EfficientNet-B0

**代码说明**：
- 同样训练 5 个 epoch，学习率 0.001。
- EfficientNet-B0 在精度和效率之间取得了很好的平衡。

**预期结果**：
- 测试准确率应接近 100%
- 训练曲线保存到 `output/EfficientNet-B0_training_curves.png`

In [ ]:
# 训练 EfficientNet-B0
print("Training EfficientNet-B0...")
history_efficientnet = train_model(
    model_efficientnet, train_loader, test_loader,
    num_epochs=5, learning_rate=0.001, model_name='EfficientNet-B0'
)

# 绘制训练曲线
plot_training_curves(history_efficientnet, 'EfficientNet-B0')

In [ ]:
# 评估 EfficientNet-B0
efficientnet_inference_time = measure_inference_time(model_efficientnet, test_loader)
efficientnet_test_acc = history_efficientnet['test_acc'][-1]
print(f"EfficientNet-B0 test accuracy     : {efficientnet_test_acc*100:.1f}%")
print(f"EfficientNet-B0 inference time    : {efficientnet_inference_time:.2f} ms")
print(f"EfficientNet-B0 training time     : {history_efficientnet['training_time']:.2f} s")

## 九、四种网络对比分析

### 9.1 对比指标说明

将 4 种网络在以下 5 个维度进行对比：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">单位</th>
</tr>
<tr>
<td style="text-align: left;"><strong>Parameters</strong></td>
<td style="text-align: left;">可训练参数总量</td>
<td style="text-align: left;">M（百万）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Model Size</strong></td>
<td style="text-align: left;">模型 state_dict 存储大小</td>
<td style="text-align: left;">MB</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Training Time</strong></td>
<td style="text-align: left;">5 个 epoch 的总训练时间</td>
<td style="text-align: left;">s（秒）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Inference Time</strong></td>
<td style="text-align: left;">单次推理平均时间</td>
<td style="text-align: left;">ms（毫秒）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Test Accuracy</strong></td>
<td style="text-align: left;">测试集分类准确率</td>
<td style="text-align: left;">%</td>
</tr>
</table>

**代码说明**：
- 汇总 4 种网络的各项指标。
- 打印对比表格。
- 绘制 5 组柱状图直观对比。
- 所有图表保存到 `output` 目录。

**预期结果**：
- ResNet18 参数最多，ShuffleNetV2 参数最少
- 轻量化网络（MobileNet、ShuffleNet）的参数量和模型体积远小于 ResNet18
- 所有网络的测试准确率都应接近 100%（4 张图片分类简单）
- 对比图保存到 `output/comparison_all_metrics.png`

In [ ]:
# 汇总 4 种网络的指标
model_names = ['ResNet18', 'MobileNetV2', 'ShuffleNetV2', 'EfficientNet-B0']

results = {
    'Parameters (M)':    [resnet_params/1e6, mobilenet_params/1e6,
                          shufflenet_params/1e6, efficientnet_params/1e6],
    'Model Size (MB)':   [resnet_size, mobilenet_size, shufflenet_size,
                          efficientnet_size],
    'Training Time (s)': [history_resnet['training_time'], history_mobilenet['training_time'],
                          history_shufflenet['training_time'], history_efficientnet['training_time']],
    'Inference Time (ms)': [resnet_inference_time, mobilenet_inference_time,
                            shufflenet_inference_time, efficientnet_inference_time],
    'Test Accuracy (%)': [resnet_test_acc*100, mobilenet_test_acc*100,
                          shufflenet_test_acc*100, efficientnet_test_acc*100],
}

# 打印对比表格
print("=" * 90)
print(f"{'Model':<20}", end='')
for key in results:
    print(f"{key:>14}", end='')
print()
print("=" * 90)
for i, name in enumerate(model_names):
    print(f"{name:<20}", end='')
    for key in results:
        print(f"{results[key][i]:>14.2f}", end='')
    print()
print("=" * 90)

# 保存对比结果到文本文件
with open('./output/comparison_results.txt', 'w') as f:
    f.write("=" * 90 + "\n")
    f.write(f"{'Model':<20}")
    for key in results:
        f.write(f"{key:>14}")
    f.write("\n")
    f.write("=" * 90 + "\n")
    for i, name in enumerate(model_names):
        f.write(f"{name:<20}")
        for key in results:
            f.write(f"{results[key][i]:>14.2f}")
        f.write("\n")
    f.write("=" * 90 + "\n")
print("\nComparison results saved to ./output/comparison_results.txt")

In [ ]:
# 绘制 5 组柱状图对比
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
metrics = list(results.keys())

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    values = results[metric]
    bars = ax.bar(model_names, values, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric, fontsize=11)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    # 在柱子上标注数值
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

# 隐藏最后一个空子图
axes[5].axis('off')

plt.suptitle('Comparison of 4 Networks for Cat-Dog Classification', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./output/comparison_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved to ./output/comparison_all_metrics.png")

### 9.2 参数量与模型体积对比

**代码说明**：
- 专门绘制参数量和模型体积的对比图，更直观展示轻量化网络的压缩效果。
- 以 ResNet18 为基准，计算其他网络的压缩比。

**预期结果**：
- 轻量化网络的参数量约为 ResNet18 的 1/3 到 1/10
- ShuffleNetV2 的模型体积最小

In [ ]:
# 参数量和模型体积对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 参数量对比
params_m = [p/1e6 for p in [resnet_params, mobilenet_params, shufflenet_params,
                             efficientnet_params]]
bars1 = ax1.bar(model_names, params_m, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Parameters Comparison', fontsize=13, fontweight='bold')
ax1.set_ylabel('Parameters (Million)', fontsize=11)
ax1.tick_params(axis='x', rotation=30, labelsize=9)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, params_m):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.2f}M', ha='center', va='bottom', fontsize=9)

# 模型体积对比
sizes = [resnet_size, mobilenet_size, shufflenet_size, efficientnet_size]
bars2 = ax2.bar(model_names, sizes, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_title('Model Size Comparison', fontsize=13, fontweight='bold')
ax2.set_ylabel('Model Size (MB)', fontsize=11)
ax2.tick_params(axis='x', rotation=30, labelsize=9)
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, sizes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.1f}MB', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('./output/comparison_params_size.png', dpi=150, bbox_inches='tight')
plt.show()

# 打印压缩比
print("\nCompression ratio (relative to ResNet18):")
print("-" * 50)
for name, param, size in zip(model_names, params_m, sizes):
    param_ratio = param / params_m[0]
    size_ratio = size / sizes[0]
    print(f"  {name:<20}: params={param_ratio:.2f}x, size={size_ratio:.2f}x")

### 9.3 训练时间与推理速度对比

**代码说明**：
- 绘制训练时间和推理时间的对比图。
- 轻量化网络不仅参数少，训练和推理也应更快。

**预期结果**：
- ShuffleNetV2 的训练时间和推理时间最短
- ResNet18 的训练和推理时间最长

In [ ]:
# 训练时间和推理时间对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 训练时间对比
train_times = [history_resnet['training_time'], history_mobilenet['training_time'],
               history_shufflenet['training_time'], history_efficientnet['training_time']]
bars1 = ax1.bar(model_names, train_times, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Training Time Comparison (5 epochs)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Training Time (seconds)', fontsize=11)
ax1.tick_params(axis='x', rotation=30, labelsize=9)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, train_times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.1f}s', ha='center', va='bottom', fontsize=9)

# 推理时间对比
infer_times = [resnet_inference_time, mobilenet_inference_time, shufflenet_inference_time,
               efficientnet_inference_time]
bars2 = ax2.bar(model_names, infer_times, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_title('Inference Time Comparison', fontsize=13, fontweight='bold')
ax2.set_ylabel('Inference Time (ms)', fontsize=11)
ax2.tick_params(axis='x', rotation=30, labelsize=9)
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, infer_times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:.2f}ms', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('./output/comparison_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Time comparison chart saved to ./output/comparison_time.png")

### 9.4 综合分析与结论

根据以上实验结果，我们可以得出以下分析结论：

**1. 参数量与模型体积**：
- ResNet18 参数最多（~11.7M），作为基准模型。
- 3 种轻量化网络参数量显著减少：ShuffleNetV2（~1.4M）最少，约为 ResNet18 的 1/8。
- MobileNetV2（~3.5M）和 EfficientNet-B0（~5.3M）参数量适中，在精度和效率间取平衡。

**2. 训练与推理效率**：
- 轻量化网络的训练时间和推理时间普遍短于 ResNet18。
- ShuffleNetV2 在推理速度上优势明显，适合实时性要求高的端侧场景。

**3. 分类精度**：
- 由于本实验仅使用 4 张图片且采用预训练权重微调，4 种网络的测试准确率都接近 100%。
- 在更复杂的数据集上（如 ImageNet），EfficientNet-B0 通常精度最高，ResNet18 次之。

**4. 适用场景建议**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">网络</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">ResNet18</td>
<td style="text-align: left;">通用基线、精度优先、计算资源充足</td>
</tr>
<tr>
<td style="text-align: left;">MobileNetV2</td>
<td style="text-align: left;">移动端、嵌入式设备、精度与速度平衡</td>
</tr>
<tr>
<td style="text-align: left;">ShuffleNetV2</td>
<td style="text-align: left;">端侧实时推理、计算资源极度受限</td>
</tr>
<tr>
<td style="text-align: left;">EfficientNet-B0</td>
<td style="text-align: left;">精度与效率最优平衡、资源中等</td>
</tr>
</table>

**5. 轻量化设计思想总结**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">使用网络</th>
<th style="text-align: left;">核心思想</th>
</tr>
<tr>
<td style="text-align: left;">残差连接</td>
<td style="text-align: left;">ResNet</td>
<td style="text-align: left;">通过 shortcut 缓解梯度消失</td>
</tr>
<tr>
<td style="text-align: left;">深度可分离卷积</td>
<td style="text-align: left;">MobileNet</td>
<td style="text-align: left;">将标准卷积拆分为 depthwise + pointwise</td>
</tr>
<tr>
<td style="text-align: left;">通道重排 + 分组卷积</td>
<td style="text-align: left;">ShuffleNet</td>
<td style="text-align: left;">分组计算减少计算量，通道重排实现信息融合</td>
</tr>
<tr>
<td style="text-align: left;">复合缩放</td>
<td style="text-align: left;">EfficientNet</td>
<td style="text-align: left;">统一缩放深度、宽度、分辨率</td>
</tr>
</table>


## 十、课后练习

请根据本实验内容完成以下题目进行自测，检验你对轻量化深度学习网络的掌握程度。

**第1题**（单选题）ResNet 的核心创新是什么？

- A. 深度可分离卷积
- B. 残差连接（Residual Connection）
- C. 通道重排
- D. 复合缩放

In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）MobileNetV2 使用的核心卷积操作是？

- A. 标准卷积
- B. 空洞卷积
- C. 深度可分离卷积
- D. 分组卷积

In [ ]:
q2 = ''  # 填入你的选项，如 'C'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）ShuffleNet 中通道重排（Channel Shuffle）的作用是？

- A. 减少参数量
- B. 实现分组卷积间的信息融合
- C. 增加模型深度
- D. 替代池化层

In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）EfficientNet 的复合缩放同时缩放哪些维度？

- A. 仅深度
- B. 仅宽度
- C. 深度、宽度、分辨率
- D. 仅分辨率

In [ ]:
q4 = ''  # 填入你的选项，如 'C'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）在 4 种网络中，参数量最少的是？

- A. ResNet18
- B. MobileNetV2
- C. ShuffleNetV2
- D. EfficientNet-B0

In [ ]:
q5 = ''  # 填入你的选项，如 'C'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）深度可分离卷积将标准卷积拆分为哪两步？

- A. Depthwise（逐通道卷积）+ Pointwise（1×1 卷积）
- B. 编码 + 解码
- C. 正向 + 反向
- D. 压缩 + 扩展

In [ ]:
q6 = ''  # 填入你的选项，如 'A'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）迁移学习中，替换预训练模型的哪一层来适应新任务？

- A. 第一个卷积层
- B. 最后的分类层
- C. 中间的池化层
- D. 所有层

In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）MobileNetV2 的倒残差结构是？

- A. 胖→瘦→胖
- B. 瘦→胖→瘦
- C. 仅胖
- D. 仅瘦

In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）本实验中，为什么使用数据增强？

- A. 提高模型精度上限
- B. 从少量图片扩充训练样本，防止过拟合
- C. 减少模型参数
- D. 加速推理

In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_01 import grade
grade(globals())

## 参考资料

- [ResNet: Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)（He et al., 2015）
- [MobileNetV2: Inverted Residuals and Linear Bottlenecks](https://arxiv.org/abs/1801.04381)（Sandler et al., 2018）
- [ShuffleNetV2: Practical Guidelines for Efficient CNN Architecture Design](https://arxiv.org/abs/1807.11164)（Ma et al., 2018）
- [EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks](https://arxiv.org/abs/1905.11946)（Tan & Le, 2019）
- [PyTorch torchvision Models Documentation](https://pytorch.org/vision/stable/models.html)
- [昇腾 NPU 开发文档](https://hiascend.com/document)

---

> **实验完成！** 通过本实验，你学习了 ResNet18 及 3 种轻量化网络的设计原理，并在昇腾 NPU 上完成了猫狗分类的完整训练与对比分析。请完成课后练习检验学习效果。